In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve().parents[1] 
sys.path.append(str(ROOT))

FILE = ROOT / "data/processed/ah_decomposition.xlsx"

In [ ]:
df = pd.read_excel(
    FILE, sheet_name='England', skiprows=5, usecols='A:H',
    names=['fy', 'net_add', 'newbuild', 'ah', 'priv_placeholder', 'blank', 's106', 'other_placeholder']
)
df = df[df.fy.notna()].copy()

# 'newbuild' is '[x]' pre-2006-07
df['newbuild'] = pd.to_numeric(df['newbuild'], errors='coerce')

# Reconstruct the two residual series exactly as the workbook does
df['priv']  = df['newbuild'] - df['ah']   # E = C - D
df['other'] = df['ah'] - df['s106']       # H = D - G

# Restrict to the usable window (2006-07 onward, where newbuild is observed)
d = df.dropna(subset=['priv']).reset_index(drop=True)
print(f"n = {len(d)}")

# Levels correlation
print("corr(priv, s106)  :", round(d['priv'].corr(d['s106']), 3))
print("corr(priv, other) :", round(d['priv'].corr(d['other']), 3))

# First-differenced correlation (removes shared trend/level drift)
print("corr(Δpriv, Δs106) :", round(d['priv'].diff().corr(d['s106'].diff()), 3))
print("corr(Δpriv, Δother):", round(d['priv'].diff().corr(d['other'].diff()), 3))

# S106 share of affordable new build, for the trend chart
d['s106_share_of_ah'] = 100 * d['s106'] / d['ah']
print(d[['fy', 's106_share_of_ah']].round(1))